# C2 - PILOT MODEL (step 9)

**Status: PILOT. Not validation.** Data come from a small anonymous pilot (classmates, 3 sessions each) collected with the C2 demo's *Pilot data* tab. The label is the person's **own rating** of how ready they feel (High / Moderate / Low / At Risk). This is perceived capacity, not a clinical or objective measure.

Rules fixed before looking at results:
1. People are never split between train and test (participant-grouped cross-validation). The participant code is never a feature.
2. Main target is **binary: Low or At Risk (1) vs High or Moderate (0)**, because the pilot is too small for four reliable classes. Four-class results are shown for information only.
3. A shuffled-label control is run for every model. "Clearly above chance" only if `mean AUC > shuffled mean + 2 x shuffled std`.
4. Personal-baseline features use **only earlier sessions of the same person** (never the current or later ones). Rows with no earlier session are dropped for those models (no imputation, so "first session" cannot leak).
5. The self-rating is the label, so it is **not** a feature. No synthetic data are mixed in.
6. Report accuracy, precision, recall, F1, balanced accuracy, AUC and the confusion matrix, never accuracy alone.

## Cell 1 - Load the pilot file
Put `c2_pilot_data.csv` in the Drive folder `AnchorAI_c2_outputs`, or upload it when asked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
OUT_DIR.mkdir(exist_ok=True)
f = OUT_DIR / 'c2_pilot_data.csv'
if not f.exists():
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    f = Path('/content') / name
raw = pd.read_csv(f)
print('file:', f, '| rows:', len(raw), '| people:', raw['participant'].nunique())
raw.head()

## Cell 2 - Data quality check
Looks for repeated rows, impossible values, people with a single session, and odd focus-check results (for example tapping constantly).

In [ ]:
df = raw.copy()
df['participant'] = df['participant'].astype(str).str.strip().str.upper()
df['time'] = pd.to_datetime(df['time'])
df = df.drop_duplicates(subset=['participant', 'time']).sort_values(['participant', 'time']).reset_index(drop=True)
df['session_order'] = df.groupby('participant').cumcount() + 1

print('rows after removing exact duplicates:', len(df))
print('\nSessions per person:')
print(df.groupby('participant').size().value_counts().sort_index().rename('people').to_string())

print('\nLabel counts:')
print(df['label'].value_counts().to_string())

checks = {
    'sleep_h outside 0-16': ~df['sleep_h'].between(0, 16),
    'min_since_break outside 0-600': ~df['min_since_break'].between(0, 600),
    'accuracy < 50%': df['accuracy_pct'] < 50,
    'premature > 40% (maybe tapping constantly)': df['premature_pct'] > 40,
    'rt < 150 ms (too fast to be real)': df['rt_ms'] < 150,
}
print('\nSuspicious rows:')
for k, m in checks.items():
    print(f'  {k}: {int(m.sum())}')
df['suspect'] = pd.concat(checks.values(), axis=1).any(axis=1)
print('rows flagged:', int(df['suspect'].sum()), 'of', len(df))
print('\nPer-person label variety (a person who always gives the same label adds little):')
print(df.groupby('participant')['label'].nunique().value_counts().sort_index().rename('people').to_string())

## Cell 3 - Decide what to keep
`DROP_SUSPECT = True` removes flagged rows (documented in the report). Change it and re-run to see if the conclusion changes.

In [ ]:
DROP_SUSPECT = True
d = df[~df['suspect']].copy() if DROP_SUSPECT else df.copy()
d['y_low'] = d['label'].isin(['Low', 'At Risk']).astype(int)
d['y4'] = d['label'].map({'High': 0, 'Moderate': 1, 'Low': 2, 'At Risk': 3})
print('rows used:', len(d), '| people:', d['participant'].nunique())
print('binary target (1 = Low or At Risk):', d['y_low'].value_counts().to_dict())
minority = d['y_low'].value_counts().min()
if len(d) < 25 or d['participant'].nunique() < 6 or minority < 8:
    print('\nWARNING: very small pilot (need roughly 25 rows, 6 people, 8 of the rarer class). Treat every number below as descriptive only.')

## Cell 4 - Features
- **Raw features** (known at the time of the session): sleep, minutes since break, five focus-check numbers, hour of day, weekend flag.
- **Personal-baseline (`dev_`) features:** today's value minus the mean of the **earlier** sessions of the same person, divided by that person's earlier spread (floored so a tiny early spread cannot explode). Needs at least 1 earlier session, so each person's first session has no `dev_` and is dropped for those models. With only 3 sessions per person the baseline is very rough (the spread of one session is unknown, so the floor is used).
- The self-rating and the participant code are never features.

In [ ]:
RAW = ['sleep_h', 'min_since_break', 'rt_ms', 'rt_sd_ms', 'miss_pct', 'premature_pct', 'accuracy_pct', 'drift_ms']
d = d.sort_values(['participant', 'time']).reset_index(drop=True)
d['hour'] = pd.to_datetime(d['time']).dt.hour
d['weekend'] = pd.to_datetime(d['time']).dt.dayofweek.isin([5, 6]).astype(int)
FEATS_RAW = RAW + ['hour', 'weekend']

# floor for the spread of each feature = 25% of its overall spread
floor = {c: max(0.25 * d[c].std(), 1e-6) for c in RAW}
dev_cols = []
for c in RAW:
    out = np.full(len(d), np.nan)
    for p, g in d.groupby('participant'):
        idx = g.index.to_numpy(); vals = g[c].to_numpy()
        for k in range(len(idx)):
            prev = vals[:k]
            if len(prev) >= 1:
                out[idx[k]] = (vals[k] - prev.mean()) / max(prev.std(), floor[c])
    d['dev_' + c] = out
    dev_cols.append('dev_' + c)
FEATS_DEV = dev_cols
d_dev = d.dropna(subset=FEATS_DEV).copy()
print('rows with a personal baseline (2nd session onward):', len(d_dev), 'of', len(d), '| people:', d_dev['participant'].nunique())
print('binary target in that subset:', d_dev['y_low'].value_counts().to_dict())

## Cell 5 - Cross-validation helper (participant-grouped, shuffled-label control)
Out-of-fold predictions are pooled per repeat, then metrics are computed once per repeat and averaged over 20 different fold layouts.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

def make_model(name):
    if name == 'Majority':
        return DummyClassifier(strategy='prior')
    if name == 'LR':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1))])

REPEATS = 20
def run_cv(data, cols, model_name, shuffled=False):
    X, y, g = data[cols].to_numpy(), data['y_low'].to_numpy(), data['participant'].to_numpy()
    k = int(min(5, len(np.unique(g))))
    rows, cms = [], []
    for s in range(REPEATS):
        yy = np.random.RandomState(500 + s).permutation(y) if shuffled else y
        prob = np.full(len(yy), np.nan)
        cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=s)
        for tr, te in cv.split(X, yy, g):
            if len(np.unique(yy[tr])) < 2:
                prob[te] = yy[tr].mean(); continue
            m = make_model(model_name).fit(X[tr], yy[tr])
            prob[te] = m.predict_proba(X[te])[:, 1]
        ok = ~np.isnan(prob)
        pred = (prob[ok] >= 0.5).astype(int)
        yt = yy[ok]
        auc = roc_auc_score(yt, prob[ok]) if len(np.unique(yt)) == 2 else np.nan
        rows.append([accuracy_score(yt, pred), precision_score(yt, pred, zero_division=0), recall_score(yt, pred, zero_division=0),
                     f1_score(yt, pred, zero_division=0), balanced_accuracy_score(yt, pred), auc])
        cms.append(confusion_matrix(yt, pred, labels=[0, 1]))
    return np.array(rows, dtype=float), np.mean(cms, axis=0)

RES, CMS = [], {}
def evaluate(name, data, cols, model_name):
    real, cm = run_cv(data, cols, model_name)
    if model_name == 'Majority':
        real[:, 5] = 0.5
    shuf = real if model_name == 'Majority' else run_cv(data, cols, model_name, shuffled=True)[0]
    thr = max(np.nanmean(shuf[:, 5]), 0.5) + 2 * np.nanstd(shuf[:, 5])
    RES.append({'experiment': name, 'model': model_name, 'rows': len(data), 'people': data['participant'].nunique(), 'features': len(cols),
                'accuracy': real[:, 0].mean(), 'precision': real[:, 1].mean(), 'recall': real[:, 2].mean(), 'F1': real[:, 3].mean(),
                'balanced acc': real[:, 4].mean(), 'AUC': np.nanmean(real[:, 5]), 'AUC std': np.nanstd(real[:, 5]),
                'shuffled AUC': np.nanmean(shuf[:, 5]),
                'clearly above chance?': ('n/a' if model_name == 'Majority' else ('YES' if np.nanmean(real[:, 5]) > thr else 'no'))})
    CMS[(name, model_name)] = cm
print('helper ready')

## Cell 6 - Experiments
- **P1 raw, all sessions:** focus check + sleep + break + time of day.
- **P2 raw, subset with a baseline:** same features on the rows that also have a personal baseline, so P2, P3 and P4 compare on the same rows.
- **P3 personal baseline (`dev_`) only:** the C2 idea that a person's own usual range matters.
- **P4 raw + `dev_`:** both.
- **P5 focus check only** and **P6 context only** (sleep, break, time): which signal group carries anything.

In [ ]:
SETS = [
    ('P1 raw, all sessions', d, FEATS_RAW),
    ('P2 raw, subset with baseline', d_dev, FEATS_RAW),
    ('P3 personal baseline only', d_dev, FEATS_DEV),
    ('P4 raw + personal baseline', d_dev, FEATS_RAW + FEATS_DEV),
    ('P5 focus check only', d, ['rt_ms', 'rt_sd_ms', 'miss_pct', 'premature_pct', 'accuracy_pct', 'drift_ms']),
    ('P6 context only', d, ['sleep_h', 'min_since_break', 'hour', 'weekend']),
]
for name, data, cols in SETS:
    if len(data) < 12 or data['y_low'].nunique() < 2 or data['participant'].nunique() < 3:
        print('skipped (too little data):', name); continue
    for m in ['Majority', 'LR', 'RF']:
        evaluate(name, data, cols, m)
res = pd.DataFrame(RES)
show = ['experiment', 'model', 'rows', 'people', 'accuracy', 'precision', 'recall', 'F1', 'balanced acc', 'AUC', 'shuffled AUC', 'clearly above chance?']
print(res[show].round(3).to_string(index=False))
res.round(4).to_csv(OUT_DIR / 'pilot_model_results.csv', index=False)

## Cell 7 - Confusion matrices and picture
Rows = real class, columns = predicted (0 = High or Moderate, 1 = Low or At Risk). Counts are averaged over 20 repeats.

In [ ]:
import matplotlib.pyplot as plt
best = res[res['model'] != 'Majority'].sort_values('AUC', ascending=False).head(3)
for _, r in best.iterrows():
    cm = CMS[(r['experiment'], r['model'])]
    print(f"{r['experiment']} / {r['model']}  (AUC {r['AUC']:.2f}, balanced acc {r['balanced acc']:.2f})")
    print(pd.DataFrame(cm.round(1), index=['true 0 (High/Mod)', 'true 1 (Low/At Risk)'], columns=['pred 0', 'pred 1']).to_string(), '\n')

rf = res[res['model'] == 'RF'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(rf))
ax.barh(y, rf['AUC'], xerr=rf['AUC std'], color='tab:blue', alpha=0.85, label='real labels')
ax.scatter(rf['shuffled AUC'], y, color='k', zorder=3, label='shuffled labels')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(y); ax.set_yticklabels(rf['experiment'], fontsize=8); ax.invert_yaxis()
ax.set_xlim(0.2, 1.0); ax.set_xlabel('ROC-AUC (Random Forest, participant-grouped CV)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / 'pilot_model_results.png', dpi=120); plt.show()

## Cell 8 - What the features do
Standardised Logistic Regression coefficients fitted on **all** pilot rows. This describes this small sample, it is not proof.

In [ ]:
lr = make_model('LR').fit(d[FEATS_RAW], d['y_low'])
coef = pd.Series(lr.named_steps['clf'].coef_[0], index=FEATS_RAW).sort_values()
print('Positive = pushes towards "Low or At Risk":')
print(coef.round(2).to_string())
plt.figure(figsize=(6, 3.5)); coef.plot.barh(color=['tab:green' if v < 0 else 'tab:red' for v in coef])
plt.axvline(0, color='k', linewidth=0.8); plt.tight_layout(); plt.savefig(OUT_DIR / 'pilot_lr_coefficients.png', dpi=120); plt.show()

## Cell 9 - The demo's provisional rule engine vs the pilot labels
The demo's rule engine (provisional weights and usual ranges) is applied to the signals the pilot also measured. It is **not trained on this data**, so this is an honest first check of the rule engine. Signals the pilot did not collect (steps, phone use at night, heart rate, app switches) are left out and the weights renormalised, as the demo does.

In [ ]:
W = {'sleep_h': (18, -1, 6.75, 0.7), 'min_since_break': (8, 1, 55, 25), 'rt_ms': (15, 1, 440, 45), 'rt_sd_ms': (12, 1, 65, 15),
     'miss_pct': (8, 1, 3, 2), 'premature_pct': (6, 1, 2, 1.5), 'accuracy_pct': (10, -1, 91, 4), 'drift_ms': (8, 1, 15, 20)}
def rule_capacity(row):
    s = w = 0.0
    for c, (wt, dr, mu, sd) in W.items():
        harm = dr * (row[c] - mu) / sd
        s += wt * min(max(harm, 0), 3) / 3; w += wt
    return 100 * (1 - s / w)
d['rule_cap'] = d.apply(rule_capacity, axis=1)
d['rule_band'] = pd.cut(d['rule_cap'], [-1, 32, 52, 72, 101], labels=['At Risk', 'Low', 'Moderate', 'High'], right=False)
auc_rule = roc_auc_score(d['y_low'], -d['rule_cap']) if d['y_low'].nunique() == 2 else np.nan
print('AUC of rule capacity vs the "Low or At Risk" rating (all rows, no training, population defaults):', round(auc_rule, 3))
print('\nBand from the rule engine (rows) vs the own rating (columns):')
print(pd.crosstab(d['rule_band'], d['label']).to_string())
print('\nNote: these usual ranges were set for one simulated person. The personal baseline (cell 6, P3) is the intended design.')

## Cell 10 - How to read this (put this in the report)
- A model is **above chance** only if `clearly above chance? = YES` **and** its AUC beats the majority baseline. With a handful of people the AUC varies a lot, so one "YES" is a hint, not a result.
- **P3 above P2 on the same rows** would support the personal-baseline idea. If it does not, say so.
- **P5 vs P6** shows whether the focus check or the context (sleep, break, time) carries the signal.
- **The label is the person's own rating.** The model learns to predict how people say they feel, not their true capacity.
- **Small group, short period, typed-in sleep, informal consent.** Write "pilot, not validated" on every slide that shows a number from this notebook.
- If nothing is above chance, that is a valid finding for a 3-session pilot. Report it and use it to justify a longer data collection.